In [1]:
import numpy as np
import pandas as pd

In [2]:
data_ids = [361260, 361254, 361259, 361253, 361243, 361242]
n_ests = [50, 100, 500, 1000]
min_samples_leafs = [1, 5, 10]
max_features = [0.1, 0.33, "1.0"]

In [3]:
# for each data_id, load the result and save as a df
dfs = []
for data_id in data_ids:
    # get number of samples in the data_id by reading X csv
    X = np.loadtxt(f"data/{data_id}/X.csv", delimiter=",")
    n_samples = 1000
    n_features = X.shape[1]
    for n_est in n_ests:
        for min_samples_leaf in min_samples_leafs:
            for max_feature in max_features:
                # create the directory if it doesn't exist
                dir_path = f"results/{data_id}/n_estimators_{n_est}/min_samples_leaf_{min_samples_leaf}/max_features_{max_feature}"
                results_path = f"{dir_path}/runtime_results.csv"
                results_df = pd.read_csv(results_path)
                # divide every col in df except 'data_id' by n_samples
                for col in results_df.columns:
                    if col != 'data_id':
                        results_df[col] = results_df[col] / n_samples
                # add columns for n_estimators, min_samples_leaf, max_features
                results_df['n_estimators'] = n_est
                results_df['min_samples_leaf'] = min_samples_leaf
                results_df['max_features'] = max_feature
                results_df['num_features'] = n_features
                dfs.append(results_df)
df = pd.concat(dfs, ignore_index=True)

In [4]:
df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])

,data_id,rf_fitting_time,rf_plus_baseline_fitting_time,rf_plus_fitting_time,shap_explainer_time,shap_values_time,lime_time,lmdi_baseline_explainer_time,lmdi_baseline_values_time,lmdi_plus_explainer_time,lmdi_plus_values_time,n_estimators,min_samples_leaf,max_features,num_features
190,361242,0.001259,0.006129,0.013413,0.000009,0.004067,0.248645,0.000007,0.028118,0.000008,0.037338,100,1,0.33,81
154,361243,0.002054,0.007530,0.017108,0.000011,0.004057,0.312608,0.000006,0.036457,0.000008,0.052858,100,1,0.33,116
118,361253,0.000767,0.005829,0.006307,0.000008,0.003174,0.130858,0.000005,0.009053,0.000011,0.011583,100,1,0.33,48
46,361254,0.000397,0.009422,0.010476,0.000009,0.002703,0.111353,0.000006,0.004555,0.000011,0.005024,100,1,0.33,21
82,361259,0.000584,0.005796,0.007990,0.000012,0.002793,0.103908,0.000005,0.005328,0.000010,0.006307,100,1,0.33,32
10,361260,0.000278,0.007364,0.008302,0.000010,0.002696,0.092050,0.000006,0.003170,0.000012,0.003226,100,1,0.33,15
193,361242,0.001206,0.006759,0.011991,0.000009,0.003786,0.224189,0.000007,0.022085,0.000007,0.029978,100,5,0.33,81
157,361243,0.001864,0.008848,0.017441,0.000009,0.003834,0.309116,0.000006,0.036889,0.000008,0.055723,100,5,0.33,116
121,361253,0.001029,0.007522,0.008766,0.000009,0.004231,0.184714,0.000008,0.013566,0.000017,0.016411,100,5,0.33,48
49,361254,0.000465,0.009515,0.013244,0.000013,0.003311,0.112093,0.000007,0.004385,0.000014,0.004794,100,5,0.33,21


In [5]:
display_df = df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, lime_time, shap_values_time, rf_plus_fitting_time + lmdi_plus_values_time
display_df = display_df[['data_id', 'num_features', 'min_samples_leaf', 'lime_time', 'shap_values_time', 'rf_plus_fitting_time', 'lmdi_plus_values_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_values_time'], inplace=True)
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lime_time': 'LIME',
    'shap_values_time': 'TreeSHAP',
    'lmdi_plus_time': 'LMDI+'
})
display_df

,OpenML Data ID,# of Features,Min. Samples per Leaf,LIME,TreeSHAP,LMDI+
190,361242,81,1,0.248645,0.004067,0.050751
154,361243,116,1,0.312608,0.004057,0.069966
118,361253,48,1,0.130858,0.003174,0.017890
46,361254,21,1,0.111353,0.002703,0.015501
82,361259,32,1,0.103908,0.002793,0.014298
10,361260,15,1,0.092050,0.002696,0.011527
193,361242,81,5,0.224189,0.003786,0.041968
157,361243,116,5,0.309116,0.003834,0.073164
121,361253,48,5,0.184714,0.004231,0.025178
49,361254,21,5,0.112093,0.003311,0.018038


In [6]:
# round to fourth decimal place
display_df = display_df.round(4)

In [7]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   Min. Samples per Leaf |   LIME |   TreeSHAP |   LMDI+ |
|-----------------:|----------------:|------------------------:|-------:|-----------:|--------:|
|           361242 |              81 |                       1 | 0.2486 |     0.0041 |  0.0508 |
|           361243 |             116 |                       1 | 0.3126 |     0.0041 |  0.07   |
|           361253 |              48 |                       1 | 0.1309 |     0.0032 |  0.0179 |
|           361254 |              21 |                       1 | 0.1114 |     0.0027 |  0.0155 |
|           361259 |              32 |                       1 | 0.1039 |     0.0028 |  0.0143 |
|           361260 |              15 |                       1 | 0.092  |     0.0027 |  0.0115 |
|           361242 |              81 |                       5 | 0.2242 |     0.0038 |  0.042  |
|           361243 |             116 |                       5 | 0.3091 |     0.0038 |  0.0732 |
|           361253 |          

In [8]:
df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5)].sort_values(by=['n_estimators', 'data_id'])

,data_id,rf_fitting_time,rf_plus_baseline_fitting_time,rf_plus_fitting_time,shap_explainer_time,shap_values_time,lime_time,lmdi_baseline_explainer_time,lmdi_baseline_values_time,lmdi_plus_explainer_time,lmdi_plus_values_time,n_estimators,min_samples_leaf,max_features,num_features
184,361242,0.001068,0.007772,0.014019,0.000011,0.003491,0.241053,0.000006,0.022669,0.000007,0.032438,50,5,0.33,81
148,361243,0.002117,0.006458,0.014406,0.000009,0.003484,0.244516,0.000005,0.030594,0.000007,0.048141,50,5,0.33,116
112,361253,0.000929,0.008899,0.012553,0.000011,0.004231,0.195321,0.000007,0.012186,0.000015,0.015183,50,5,0.33,48
40,361254,0.000425,0.008337,0.010072,0.000012,0.002641,0.107785,0.000008,0.004757,0.000017,0.005345,50,5,0.33,21
76,361259,0.000830,0.007338,0.015378,0.000012,0.003831,0.151741,0.000007,0.007403,0.000013,0.008446,50,5,0.33,32
4,361260,0.000260,0.006820,0.007924,0.000004,0.002712,0.089860,0.000005,0.003123,0.000013,0.003418,50,5,0.33,15
193,361242,0.001206,0.006759,0.011991,0.000009,0.003786,0.224189,0.000007,0.022085,0.000007,0.029978,100,5,0.33,81
157,361243,0.001864,0.008848,0.017441,0.000009,0.003834,0.309116,0.000006,0.036889,0.000008,0.055723,100,5,0.33,116
121,361253,0.001029,0.007522,0.008766,0.000009,0.004231,0.184714,0.000008,0.013566,0.000017,0.016411,100,5,0.33,48
49,361254,0.000465,0.009515,0.013244,0.000013,0.003311,0.112093,0.000007,0.004385,0.000014,0.004794,100,5,0.33,21


In [9]:
display_df = df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5) & (df['n_estimators'] != 50)].sort_values(by=['n_estimators', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, lime_time, shap_values_time, rf_plus_fitting_time + lmdi_plus_values_time
display_df = display_df[['data_id', 'num_features', 'n_estimators', 'lime_time', 'shap_values_time', 'rf_plus_fitting_time', 'lmdi_plus_values_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_values_time'], inplace=True)
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lime_time': 'LIME',
    'shap_values_time': 'TreeSHAP',
    'lmdi_plus_time': 'LMDI+'
})
display_df

,OpenML Data ID,# of Features,# of Estimators,LIME,TreeSHAP,LMDI+
193,361242,81,100,0.224189,0.003786,0.041968
157,361243,116,100,0.309116,0.003834,0.073164
121,361253,48,100,0.184714,0.004231,0.025178
49,361254,21,100,0.112093,0.003311,0.018038
85,361259,32,100,0.134084,0.003485,0.020884
13,361260,15,100,0.092258,0.003339,0.013281
202,361242,81,500,0.184918,0.002991,0.035427
166,361243,116,500,0.273784,0.003884,0.063366
130,361253,48,500,0.148719,0.003174,0.020331
58,361254,21,500,0.090191,0.002407,0.011842


In [10]:
# round to fourth decimal place
display_df = display_df.round(4)

In [11]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   # of Estimators |   LIME |   TreeSHAP |   LMDI+ |
|-----------------:|----------------:|------------------:|-------:|-----------:|--------:|
|           361242 |              81 |               100 | 0.2242 |     0.0038 |  0.042  |
|           361243 |             116 |               100 | 0.3091 |     0.0038 |  0.0732 |
|           361253 |              48 |               100 | 0.1847 |     0.0042 |  0.0252 |
|           361254 |              21 |               100 | 0.1121 |     0.0033 |  0.018  |
|           361259 |              32 |               100 | 0.1341 |     0.0035 |  0.0209 |
|           361260 |              15 |               100 | 0.0923 |     0.0033 |  0.0133 |
|           361242 |              81 |               500 | 0.1849 |     0.003  |  0.0354 |
|           361243 |             116 |               500 | 0.2738 |     0.0039 |  0.0634 |
|           361253 |              48 |               500 | 0.1487 |     0.0032 |  0.0203 |